# 04 · LLM Tabanlı Duygu Analizi (yerel model)
**Model:** Qwen2.5-7B-Instruct (açık kaynak, 4-bit, Colab T4 GPU)
API ve kota yok. Zero-shot, greedy decoding (temperature 0 ile aynı: her çalıştırmada aynı sonuç), iki dilde aynı prompt.
Geçersiz cevapta bir kez tekrar sorulur, yine olmazsa `Geçersiz` yazılır.

**Kullanım:** Çalışma zamanı → Çalışma zamanı türünü değiştir → **T4 GPU**. Sonra **Tümünü çalıştır**.
Yarıda kesilirse tekrar *Tümünü çalıştır* demen yeterli, kaldığı yerden devam eder.

In [17]:
# ==== AYARLAR ====
REPO_URL = "https://github.com/nazlicandogann/duygu-analizi.git"   # <- grubun repo adresi
REPO_DIR = "/content/repo"
DATA = {"en": "data/test_en.csv", "tr": "data/test_tr.csv"}   # id,text,label
LANGS = ["en", "tr"]
MODEL_ID = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
PILOT_PER_CLASS = None      # pilot için 20; tüm veri için None
BATCH_SIZE = 16
SEED = 42
LABELS = ["Pozitif", "Negatif", "Nötr"]

In [10]:
# ==== KURULUM: GPU kontrolü + repo ====
import os, subprocess, torch
assert torch.cuda.is_available(), "GPU yok! Çalışma zamanı türünü T4 GPU yap."
print("GPU:", torch.cuda.get_device_name(0))

token = None
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception as e:
    print("GITHUB_TOKEN okunamadı (push için gerekli):", e)

url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", url, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
os.chdir(REPO_DIR)
os.makedirs("results", exist_ok=True)
print("Çalışma klasörü:", os.getcwd())

GPU: Tesla T4
Çalışma klasörü: /content/repo


In [11]:
!pip -q install -U bitsandbytes accelerate

In [12]:
# ==== MODELİ YÜKLE (ilk seferde ~5 GB indirir, birkaç dakika) ====
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
torch.manual_seed(SEED)
tok = AutoTokenizer.from_pretrained(MODEL_ID, padding_side="left")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                           bnb_4bit_compute_dtype=torch.float16),
    device_map="auto",
)
model.eval()
print("Model hazır.")

config.json:   0%|          | 0.00/1.34k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.36k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (4 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

Model hazır.


In [13]:
# ==== PROMPT (iki dilde aynı) ====
import re
SYSTEM = "You are a sentiment classification system."
PROMPT = ('Classify the sentiment of the following text as Positive, Negative, or Neutral.\n'
          'The text may be in English or Turkish.\n'
          'Answer with exactly one word: Positive, Negative, or Neutral.\n\n'
          'Text: """{text}"""\n'
          'Sentiment:')
RETRY_SUFFIX = "\n\nIMPORTANT: Reply with ONLY one word: Positive, Negative, or Neutral."

MAP = {"positive": "Pozitif", "negative": "Negatif", "neutral": "Nötr",
       "pozitif": "Pozitif", "negatif": "Negatif", "nötr": "Nötr", "notr": "Nötr"}

def parse(ans):
    words = re.findall(r"[a-zA-ZçğıöşüÇĞİÖŞÜ]+", ans.lower())
    found = {MAP[w] for w in words if w in MAP}
    return found.pop() if len(found) == 1 else None   # tek ve net cevap değilse geçersiz

def build(text, retry=False):
    msgs = [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": PROMPT.format(text=text) + (RETRY_SUFFIX if retry else "")}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def generate(prompts):
    enc = tok(prompts, return_tensors="pt", padding=True, truncation=True, max_length=512).to(model.device)
    out = model.generate(**enc, max_new_tokens=5, do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)

print(generate([build("I love this airline!"), build("Uçuş berbattı, bir daha asla.")]))

[transformers] Both `max_new_tokens` (=5) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


['Positive', 'Negative']


In [20]:
# ==== ÇALIŞTIR (kaldığı yerden devam eder) ====
import json, time, pandas as pd

def load_data(lang):
    df = pd.read_csv(DATA[lang])
    if PILOT_PER_CLASS:
        df = (df.groupby("label", group_keys=False)
                .apply(lambda g: g.sample(min(len(g), PILOT_PER_CLASS), random_state=SEED)))
    return df.reset_index(drop=True)

def run(lang):
    tag = "_pilot" if PILOT_PER_CLASS else ""
    raw_path = f"results/llm_raw_local_{lang}{tag}.jsonl"
    df = load_data(lang)
    done = {}
    if os.path.exists(raw_path):
        with open(raw_path, encoding="utf-8") as f:
            for line in f:
                r = json.loads(line); done[str(r["id"])] = r
    todo = df[~df["id"].astype(str).isin(done)]
    print(f"[{lang}] toplam {len(df)} | hazır {len(done)} | kalan {len(todo)}")
    t0 = time.time()
    with open(raw_path, "a", encoding="utf-8") as f:
        for i in range(0, len(todo), BATCH_SIZE):
            chunk = todo.iloc[i:i + BATCH_SIZE]
            answers = generate([build(t) for t in chunk["text"]])
            for (_, row), ans in zip(chunk.iterrows(), answers):
                pred, retried = parse(ans), False
                if pred is None:                       # bir kez tekrar sor
                    ans2 = generate([build(row["text"], retry=True)])[0]
                    pred, retried, ans = parse(ans2), True, ans + " || " + ans2
                rec = {"id": str(row["id"]), "pred": pred or "Geçersiz", "raw": ans, "retried": retried}
                f.write(json.dumps(rec, ensure_ascii=False) + "\n"); done[rec["id"]] = rec
            f.flush()
            n = i + len(chunk)
            if n % (BATCH_SIZE * 20) == 0 or n == len(todo):
                print(f"  {n}/{len(todo)}  ({(time.time()-t0)/60:.1f} dk)")
    df["pred"] = df["id"].astype(str).map(lambda k: done[k]["pred"])
    out = df.rename(columns={"label": "true"})[["id", "text", "true", "pred"]]
    out_path = f"results/preds_llm_{lang}{tag}.csv"
    out.to_csv(out_path, index=False, encoding="utf-8")
    acc = (out["true"] == out["pred"]).mean()
    n_inv = (out["pred"] == "Geçersiz").sum()
    n_retry = sum(done[str(k)]["retried"] for k in df["id"])
    print(f"[{lang}] → {out_path} | accuracy {acc:.3f} | Geçersiz {n_inv} | tekrar sorulan {n_retry}")
    return out

results = {lang: run(lang) for lang in LANGS}

[transformers] Both `max_new_tokens` (=5) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[en] toplam 3000 | hazır 3000 | kalan 0
[en] → results/preds_llm_en.csv | accuracy 0.798 | Geçersiz 2 | tekrar sorulan 3
[tr] toplam 3000 | hazır 1536 | kalan 1464


[transformers] Both `max_new_tokens` (=5) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=5) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=5) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=5) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

  320/1464  (9.4 dk)


[transformers] Both `max_new_tokens` (=5) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=5) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=5) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=5) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

  640/1464  (17.5 dk)


... (2 outputs left out)

[transformers] Both `max_new_tokens` (=5) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=5) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=5) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=5) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

  1280/1464  (35.8 dk)


[transformers] Both `max_new_tokens` (=5) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=5) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=5) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=5) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

  1464/1464  (41.4 dk)
[tr] → results/preds_llm_tr.csv | accuracy 0.781 | Geçersiz 23 | tekrar sorulan 23


In [21]:
# ==== HIZLI KONTROL: sınıf bazında hatalı tahmin sayıları + confusion matrix ====
for lang, out in results.items():
    print(f"\n===== {lang.upper()} =====")
    cm = pd.crosstab(out["true"], out["pred"]).reindex(index=LABELS, columns=LABELS + ["Geçersiz"], fill_value=0)
    print(cm)
    wrong = out[out["true"] != out["pred"]].groupby("true").size().reindex(LABELS, fill_value=0)
    print("Hatalı tahmin (gerçek sınıfa göre):", wrong.to_dict(), "| toplam", int(wrong.sum()))

# Ortak değerlendirme scripti (varsa)
if os.path.exists("src/evaluate.py") and not PILOT_PER_CLASS:
    for lang in LANGS:
        !python src/evaluate.py results/preds_llm_{lang}.csv


===== EN =====
pred     Pozitif  Negatif  Nötr  Geçersiz
true                                     
Pozitif      879       33    86         2
Negatif       16      894    90         0
Nötr         186      192   622         0
Hatalı tahmin (gerçek sınıfa göre): {'Pozitif': 121, 'Negatif': 106, 'Nötr': 378} | toplam 605

===== TR =====
pred     Pozitif  Negatif  Nötr  Geçersiz
true                                     
Pozitif      800       37   160         3
Negatif       55      751   174        20
Nötr          96      111   793         0
Hatalı tahmin (gerçek sınıfa göre): {'Pozitif': 200, 'Negatif': 249, 'Nötr': 207} | toplam 656
dosya: preds_llm_en.csv
n: 3000
accuracy: 0.7983
macro_f1: 0.7935
hatalar: {'Pozitif': 121, 'Negatif': 106, 'Nötr': 378}
gecersiz: 2
gorsel: results/figures/cm_llm_en.png
dosya: preds_llm_tr.csv
n: 3000
accuracy: 0.7813
macro_f1: 0.7856
hatalar: {'Pozitif': 200, 'Negatif': 249, 'Nötr': 207}
gecersiz: 23
gorsel: results/figures/cm_llm_tr.png


In [23]:
# ==== REPOYA KAYDET (pilot değil, tüm veri bitince) ====
!git config user.name "Ceren"
!git config user.email "cerennimat@gmail.com"
!git add results/llm_raw_local_*.jsonl results/preds_llm_*.csv results/figures/cm_llm_*.png
!git commit -m "llm: Qwen2.5-7B-Instruct yerel model tahminleri"
!git push

[main 1930fd6] llm: Qwen2.5-7B-Instruct yerel model tahminleri
 2 files changed, 0 insertions(+), 0 deletions(-)
 create mode 100644 results/figures/cm_llm_en.png
 create mode 100644 results/figures/cm_llm_tr.png
Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 2 threads
Compressing objects: 100% (6/6), done.
Writing objects: 100% (6/6), 67.49 KiB | 16.87 MiB/s, done.
Total 6 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/nazlicandogann/duygu-analizi.git
   66e33ac..1930fd6  main -> main
